In [2]:
import json
from glob import glob

import pandas as pd

flores = glob('flores/devtest/*.json')
for l in flores:
    with open(l) as f:
        language = l.split('/')[-1].split('.')[0]
        with open(f'flores/{language}.txt', 'w') as out:
            data = json.load(f)
            for d in data:
                out.write(d['text'] + '\n')
                

In [12]:
from glob import glob
from pathlib import Path
langs = glob('flores/*.txt')
utf8_langs = []
for lang in langs:
    lang_name =lang.split('/')[-1].split('.')[0]
    l = 0
    sents = Path(lang).read_text().strip().split('\n')
    for sent in sents:
        l+=len(sent.encode('utf-8'))
        # l+=len(sent)
    utf8_langs.append({'lang':lang_name, 'value':l})
    

In [13]:
pd.DataFrame(utf8_langs).to_csv('langs_utf8.csv',index=False)

In [65]:
import json
from glob import glob
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

vocab_size = [8192, 32768, 49152, 65536, 81920]
metrics = ['bpb', 'cpb', 'ppl', 'sent-nll', 'mrr', 'token-nll']
random_seeds = [32, 42, 53, 67, 100, 99, 68, 69, 72, 101, 88, 77, 66, 55, 44]
sample_sizes = [1, 10, 100, 500, 1000, 1500, 1997]

for sample_size in sample_sizes:
    for random_seed in random_seeds:
        random.seed(random_seed)
        paraphrase = []
        sampled_ids = set(random.sample(range(0, 1997), sample_size)) 

        for metric in metrics:
            for vocab in vocab_size:
                en= pd.read_csv(f'ppl_results_{metric}_en/gpt2_small_EN_bpe_{vocab}_parallel10_42.csv').to_dict(orient='records')
                de= pd.read_csv(f'ppl_results_{metric}_de/gpt2_small_DE_bpe_{vocab}_parallel10_42.csv').to_dict(orient='records')
                de_ar = pd.read_csv(f'ppl_results_{metric}_de-ar/gpt2_small_DE_bpe_{vocab}_parallel10_42.csv').to_dict(orient='records')
                de_p  = pd.read_csv(f'ppl_results_{metric}_de-p/gpt2_small_DE_bpe_{vocab}_parallel10_42.csv').to_dict(orient='records')

                sent_e_list   = []
                sent_d_list   = []
                sent_d_p_list = []
                lowest        = []
                highest       = []

                for i, (e, d, d_ar, d_p) in enumerate(zip(en, de, de_ar, de_p)):
                    if i not in sampled_ids:
                        continue
                    sent_e_list.append(e['Epoch-10'])
                    sent_d_list.append(d['Epoch-10'])
                    sent_d_p_list.append(d_p['Epoch-10'])
                    lowest.append(min(d['Epoch-10'], d_ar['Epoch-10']))
                    highest.append(max(d['Epoch-10'], d_ar['Epoch-10']))

                if metric == 'ppl':
                    sent_e_ave    = np.exp(np.mean(np.log(sent_e_list)))
                    sent_d_ave    = np.exp(np.mean(np.log(sent_d_list)))
                    sent_d_p_ave  = np.exp(np.mean(np.log(sent_d_p_list)))
                    lowest_ave    = np.exp(np.mean(np.log(lowest)))
                    highest_ave   = np.exp(np.mean(np.log(highest)))
                else:
                    sent_e_ave    = np.mean(sent_e_list)
                    sent_d_ave    = np.mean(sent_d_list)
                    sent_d_p_ave  = np.mean(sent_d_p_list)
                    lowest_ave    = np.mean(lowest)
                    highest_ave   = np.mean(highest)

                paraphrase.append({'lang':'EN','tokenization':'bpe','vocab_size':vocab,'eval_data':'en',        'metric_type':metric,'mean_value':sent_e_ave,   'seed':random_seed})
                paraphrase.append({'lang':'DE','tokenization':'bpe','vocab_size':vocab,'eval_data':'de',        'metric_type':metric,'mean_value':sent_d_ave,   'seed':random_seed})
                paraphrase.append({'lang':'DE','tokenization':'bpe','vocab_size':vocab,'eval_data':'de-p',      'metric_type':metric,'mean_value':sent_d_p_ave, 'seed':random_seed})
                paraphrase.append({'lang':'DE','tokenization':'bpe','vocab_size':vocab,'eval_data':'DE_lowest', 'metric_type':metric,'mean_value':lowest_ave,   'seed':random_seed})
                paraphrase.append({'lang':'DE','tokenization':'bpe','vocab_size':vocab,'eval_data':'DE_highest','metric_type':metric,'mean_value':highest_ave,  'seed':random_seed})

        pd.DataFrame(paraphrase).to_csv(f'sample_size/paraphrase_results_{sample_size}_{random_seed}.csv', index=False)

[158]
[1309]
[1263]
[153]
[298]
[827]
[1518]
[1401]
[150]
[1190]
[814]
[1636]
[145]
[185]
[836]
[158, 1897, 437, 296, 620, 1430, 1843, 487, 1016, 49]
[1309, 228, 51, 1518, 563, 501, 457, 285, 1508, 209]
[1263, 1747, 1823, 442, 934, 1028, 1456, 988, 1918, 1470]
[153, 237, 1568, 836, 1990, 957, 850, 546, 1935, 1221]
[298, 941, 931, 1949, 1578, 357, 1444, 804, 1499, 716]
[827, 779, 409, 1227, 366, 471, 508, 272, 1556, 177]
[1518, 955, 1926, 1412, 1530, 1690, 1859, 1025, 227, 1242]
[1401, 76, 196, 1641, 341, 136, 1239, 705, 671, 1883]
[150, 1601, 1217, 1512, 381, 704, 1431, 1114, 1269, 766]
[1190, 1734, 398, 1860, 1976, 1104, 1892, 734, 956, 99]
[814, 1994, 388, 681, 1848, 378, 1281, 29, 1078, 1573]
[1636, 517, 667, 404, 492, 395, 1684, 1924, 235, 598]
[145, 638, 890, 1891, 504, 1611, 912, 602, 524, 1764]
[185, 1946, 401, 307, 1514, 619, 1722, 163, 1530, 376]
[836, 1065, 1109, 1436, 1765, 238, 361, 777, 461, 593]
[158, 1897, 437, 296, 620, 1430, 1843, 487, 1016, 49, 1475, 78, 205, 665, 104

ValueError: Sample larger than population or is negative

In [67]:
from glob import glob
for sample_size in sample_sizes:
    paraphrases = glob(f'sample_size/paraphrase_results_{sample_size}_*.csv')
    print(len(paraphrases))
    with open(f'sample_size/paraphrase_results_{sample_size}.csv', 'w') as out:
        results = []
        for p in paraphrases:
            sents = pd.read_csv(p).to_dict(orient='records')
            for s in sents:
                s['seed'] = int(p.split('/')[-1].split('.')[0].split('_')[-1])
                results.append(s)
    
        pd.DataFrame(results).to_csv(out, index=False)
        
    

15
15
15
15
15
15
15


In [71]:
from glob import glob
from tqdm import tqdm 
conllu = glob('pud/*.conllu')
for c in tqdm(conllu):
    with open(c) as f:
        language = c.split('/')[-1].split('_')[0].upper()
        if len(language) !=2:
            continue
        texts = []
        with open(f'pud/{language}.txt', 'w') as out:
            for line in f:
                if line.startswith('# text = '):
                    text = line.split('# text = ')[1].strip()
                    texts.append(text)
            print(len(texts))
            out.write('\n'.join(texts))
                
                

100%|██████████| 11/11 [00:00<00:00, 144.79it/s]

1000
1000
1000
1000
1000
1000
1000
1000
1000
1000
